# model

# This file defines the complete architecture for a mini-DeepSeek V3 model
# integrating MLA, Decoupled RoPE, MoE, MTP, and a correct KV Cache with RoPE position offsetting.

In [1]:
import gc
import torch

gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [2]:
# Import mathematical functions such as sqrt()
# Used later for scaling attention scores in self-attention
import math

# Import dataclass decorator
# Automatically creates constructor (__init__) and utility methods
from dataclasses import dataclass

# Import type hints
# Optional means value can be None
# Tuple means multiple values returned together
from typing import Optional, Tuple

# Import PyTorch library
# Provides tensors and deep learning operations
import torch

# Import neural network module
# Contains Linear, LayerNorm, Dropout, Embedding, etc.
import torch.nn as nn

# Import PyTorch functional API
# Contains softmax, gelu, cross_entropy, etc.
from torch.nn import functional as F

# Configuration Dataclass

In [3]:
# Create a dataclass to store all model hyperparameters
@dataclass
class ModelArgs:

    # Hidden dimension of the Transformer
    # Each token is represented by a vector of size 512
    d_model: int = 256

    # Number of Transformer blocks
    # More layers allow learning deeper patterns
    n_layers: int = 4

    # Vocabulary size of GPT-2 tokenizer
    # Number of unique tokens the model can predict
    vocab_size: int = 50257

    # Number of attention heads
    # Attention is split into 8 parallel heads
    num_heads: int = 4

    # Latent dimension used in MLA compression
    # Reduces KV cache memory usage
    d_latent: int = 64

    # Dimension used for Rotary Positional Encoding
    d_rope: int = 32

    # Length
    max_seq_len: int = 1024

    dropout: float = 0.1

    # Multi-Token Prediction
    n_mtp_modules: int = 1

    # MoE (only if your model uses MoE)
    moe_n_routed_experts: int = 4
    moe_n_shared_experts: int = 1
    moe_top_k: int = 2
    moe_routed_hidden: int = 256

# Rotary Positional Encoding (RoPE) Helper Module

Explanation section for the code below.


In [4]:
# Rotary Positional Encoding (RoPE) Helper Module
# This module implements Rotary Positional Encoding (RoPE).
# RoPE injects positional information into attention vectors using rotations.
# Unlike learned positional embeddings, RoPE preserves relative positions naturally.
# It is widely used in modern LLMs such as LLaMA, DeepSeek, Qwen, and Mistral.

class RotaryPositionalEncoding(nn.Module):

    # Constructor of the RoPE module
    # d_head = dimension of one attention head
    # max_seq_len = maximum sequence length supported by the model
    def __init__(self, d_head: int, max_seq_len: int = 2048):

        # Initialize parent nn.Module class
        super().__init__()

        # Store attention head dimension
        self.d_head = d_head

        # Compute RoPE frequency values
        # Generates frequencies for each pair of dimensions
        # Shape: [d_head/2]
        theta = 1.0 / (
            10000 ** (
                torch.arange(0, d_head, 2).float() / d_head
            )
        )

        # Register theta as a buffer
        # Buffers are saved with the model but are not trainable parameters
        self.register_buffer('theta', theta)

        # Generate position indices
        # Example: [0,1,2,3,...,2047]
        positions = torch.arange(max_seq_len)

        # Compute all positional frequencies
        # Shape: [max_seq_len, d_head/2]
        freqs = torch.outer(positions, self.theta)

        # Convert frequencies into complex numbers
        # Magnitude = 1
        # Angle = frequency
        # RoPE performs rotations using complex multiplication
        freqs_cis = torch.polar(
            torch.ones_like(freqs),
            freqs
        )

        # Store precomputed rotations
        # persistent=False means do not save inside checkpoints
        self.register_buffer(
            'freqs_cis',
            freqs_cis,
            persistent=False
        )

    # Forward pass
    # x shape = [Batch, Heads, Sequence_Length, Head_Dimension]
    # position_offset is used during KV Cache inference
    def forward(
        self,
        x: torch.Tensor,
        position_offset: int = 0
    ):

        # Extract sequence length
        seq_len = x.shape[2]

        # Convert tensor into complex representation
        # Every pair of values becomes one complex number
        #
        # Example:
        # [x1,x2,x3,x4]
        #
        # becomes
        #
        # [(x1+x2i),(x3+x4i)]
        #
        # Shape:
        # [B,H,S,D]
        #
        # becomes
        #
        # [B,H,S,D/2]
        x_complex = torch.view_as_complex(
            x.float().reshape(
                *x.shape[:-1],
                -1,
                2
            )
        )

        # Select required positional frequencies
        # During generation we continue from cached positions
        freqs_cis_slice = self.freqs_cis[
            position_offset:
            position_offset + seq_len
        ]

        # Add batch dimension
        # Add head dimension
        #
        # Shape:
        # [S,D/2]
        #
        # becomes
        #
        # [1,1,S,D/2]
        freqs_cis = freqs_cis_slice.unsqueeze(0).unsqueeze(0)

        # Apply rotary transformation
        # Complex multiplication performs the rotation
        x_rotated = x_complex * freqs_cis

        # Convert complex numbers back into real values
        #
        # Shape:
        # [B,H,S,D/2]
        #
        # becomes
        #
        # [B,H,S,D/2,2]
        x_out = torch.view_as_real(x_rotated)

        # Merge final dimensions
        #
        # Shape:
        # [B,H,S,D/2,2]
        #
        # becomes
        #
        # [B,H,S,D]
        x_out = x_out.flatten(3)

        # Convert output back to original datatype
        # Example:
        # float32 -> bfloat16
        # float32 -> float16
        return x_out.type_as(x)

# DeepSeekAttention

In [5]:
# --- Multi-Head Latent Attention (MLA) Module ---
# This module implements DeepSeek's Multi-Head Latent Attention (MLA).
# MLA reduces KV Cache memory by compressing Key and Value information
# into a smaller latent representation and reconstructing them when needed.
#
# Traditional Attention:
# Input → Q,K,V → Attention
#
# MLA:
# Input → Q
# Input → Compressed Latent KV
# Latent KV → Reconstructed K,V
# → Attention
#
# MLA also separates:
# 1. Content Information
# 2. Positional Information (RoPE)

class DeepSeekAttention(nn.Module):

    # Constructor
    # Initializes all layers required for MLA attention
    def __init__(self, args: ModelArgs):

        # Initialize parent PyTorch module
        super().__init__()

        # Store model hidden dimension
        self.d_model = args.d_model

        # Number of attention heads
        self.num_heads = args.num_heads

        # Dimension per attention head
        #
        # Example:
        # d_model = 512
        # num_heads = 8
        #
        # d_head = 64
        self.d_head = args.d_model // args.num_heads

        # Latent compression dimension
        # Used to reduce KV Cache memory
        self.d_latent = args.d_latent

        # RoPE positional dimension
        self.d_rope = args.d_rope


        # ==================================================
        # PATH A : CONTENT INFORMATION
        # ==================================================

        # Generate Content Query Matrix
        #
        # Input:
        # [B,S,d_model]
        #
        # Output:
        # [B,S,d_model]
        self.W_q_content = nn.Linear(
            args.d_model,
            args.d_model,
            bias=False
        )

        # Compress hidden states into latent KV representation
        #
        # Input:
        # [B,S,d_model]
        #
        # Output:
        # [B,S,d_latent]
        self.W_dkv_content = nn.Linear(
            args.d_model,
            args.d_latent,
            bias=False
        )

        # Reconstruct Keys from latent representation
        #
        # Input:
        # [B,S,d_latent]
        #
        # Output:
        # [B,S,d_model]
        self.W_uk_content = nn.Linear(
            args.d_latent,
            args.d_model,
            bias=False
        )

        # Reconstruct Values from latent representation
        #
        # Input:
        # [B,S,d_latent]
        #
        # Output:
        # [B,S,d_model]
        self.W_uv_content = nn.Linear(
            args.d_latent,
            args.d_model,
            bias=False
        )


        # ==================================================
        # PATH B : POSITION INFORMATION
        # ==================================================

        # Generate positional Keys
        #
        # Output:
        # [B,S,num_heads*d_rope]
        self.W_k_pos = nn.Linear(
            args.d_model,
            args.d_rope * args.num_heads,
            bias=False
        )

        # Generate positional Queries
        #
        # Output:
        # [B,S,num_heads*d_rope]
        self.W_q_pos = nn.Linear(
            args.d_model,
            args.d_rope * args.num_heads,
            bias=False
        )

        # Rotary Positional Encoding module
        self.rope = RotaryPositionalEncoding(
            args.d_rope,
            max_seq_len=args.max_seq_len
        )


        # ==================================================
        # OUTPUT PROJECTION
        # ==================================================

        # Converts concatenated attention output
        # back to model dimension
        self.W_o = nn.Linear(
            args.d_model,
            args.d_model,
            bias=False
        )

        # Dropout for regularization
        self.dropout = nn.Dropout(args.dropout)


    # Forward Pass
    #
    # x:
    # [B,S,D]
    #
    # attn_mask:
    # Causal mask
    #
    # past_kv:
    # Previous KV cache
    #
    # position_offset:
    # Used during autoregressive generation
    def forward(
        self,
        x: torch.Tensor,
        attn_mask: torch.Tensor,
        past_kv: Optional[
            Tuple[torch.Tensor, torch.Tensor]
        ] = None,
        position_offset: int = 0
    ):

        # Extract tensor dimensions
        B, S, D = x.shape

        # Determine cache length
        #
        # Training:
        # past_len = 0
        #
        # Inference:
        # past_len = cached sequence length
        past_len = (
            past_kv[0].shape[1]
            if past_kv is not None
            else 0
        )


        # ==================================================
        # PATH A : CONTENT ATTENTION
        # ==================================================

        # Generate content queries
        #
        # Shape:
        # [B,S,D]
        #
        # ->
        #
        # [B,H,S,D_head]
        q_c = self.W_q_content(x)\
            .view(
                B,
                S,
                self.num_heads,
                self.d_head
            )\
            .transpose(1, 2)

        # Compress hidden states
        #
        # [B,S,D]
        #
        # ->
        #
        # [B,S,d_latent]
        c_kv_new = self.W_dkv_content(x)

        # Append old KV cache
        if past_kv is not None:

            c_kv = torch.cat(
                [past_kv[0], c_kv_new],
                dim=1
            )

        else:

            c_kv = c_kv_new

        # Reconstruct content keys
        #
        # [B,S,d_latent]
        #
        # ->
        #
        # [B,H,S,D_head]
        k_c = self.W_uk_content(c_kv)\
            .view(
                B,
                past_len + S,
                self.num_heads,
                self.d_head
            )\
            .transpose(1, 2)

        # Reconstruct content values
        #
        # [B,H,S,D_head]
        v_c = self.W_uv_content(c_kv)\
            .view(
                B,
                past_len + S,
                self.num_heads,
                self.d_head
            )\
            .transpose(1, 2)


        # ==================================================
        # PATH B : POSITION ATTENTION
        # ==================================================

        # Generate positional keys
        k_r_unrotated = self.W_k_pos(x)\
            .view(
                B,
                S,
                self.num_heads,
                self.d_rope
            )\
            .transpose(1, 2)

        # Generate positional queries
        q_r_unrotated = self.W_q_pos(x)\
            .view(
                B,
                S,
                self.num_heads,
                self.d_rope
            )\
            .transpose(1, 2)

        # Apply RoPE to keys
        k_r_new = self.rope(
            k_r_unrotated,
            position_offset=position_offset
        )

        # Apply RoPE to queries
        q_r = self.rope(
            q_r_unrotated,
            position_offset=position_offset
        )

        # Append cached positional keys
        if past_kv is not None:

            k_r = torch.cat(
                [past_kv[1], k_r_new],
                dim=2
            )

        else:

            k_r = k_r_new


        # ==================================================
        # COMPUTE ATTENTION SCORES
        # ==================================================

        # Content similarity scores
        #
        # Shape:
        # [B,H,S,S]
        content_scores = (
            q_c @ k_c.transpose(-2, -1)
        ) / math.sqrt(self.d_head)

        # Positional similarity scores
        position_scores = (
            q_r @ k_r.transpose(-2, -1)
        ) / math.sqrt(self.d_rope)

        # Combine content and position information
        attn_scores = (
            content_scores +
            position_scores
        )

        # Apply causal mask
        attn_scores = attn_scores + attn_mask

        # Convert scores into probabilities
        attn_weights = F.softmax(
            attn_scores,
            dim=-1
        )

        # Apply dropout
        attn_weights = self.dropout(
            attn_weights
        )


        # ==================================================
        # COMPUTE CONTEXT VECTOR
        # ==================================================

        # Weighted sum of values
        context_vector = (
            attn_weights @ v_c
        ).transpose(1, 2)\
         .contiguous()\
         .view(B, S, D)

        # Final output projection
        output = self.W_o(
            context_vector
        )


        # ==================================================
        # SAVE NEW CACHE
        # ==================================================

        # Cache contains:
        # 1. Latent Content KV
        # 2. Positional Keys
        new_cache = (
            c_kv,
            k_r
        )

        # Return attention output
        # Return updated cache
        return output, new_cache

# Mixture-of-Experts (MoE) Modules


In [6]:
# ==================================================
# EXPERT FEED FORWARD NETWORK (Expert FFN)
# ==================================================
# Each expert is an independent Feed Forward Network.
# Different experts learn different types of knowledge.
#
# Example:
# Expert 1 → Grammar
# Expert 2 → Mathematics
# Expert 3 → Coding
# Expert 4 → Reasoning
#
# Router decides which expert(s) should process a token.

class ExpertFFN(nn.Module):

    # Constructor
    # d_model = input feature dimension
    # hidden = hidden layer dimension
    # dropout = regularization probability
    def __init__(
        self,
        d_model: int,
        hidden: int,
        dropout: float = 0.0
    ):

        # Initialize parent PyTorch module
        super().__init__()

        # First linear layer
        #
        # Shape:
        # [B,S,d_model]
        #
        # ->
        #
        # [B,S,hidden]
        self.fc1 = nn.Linear(
            d_model,
            hidden,
            bias=False
        )

        # Second linear layer
        #
        # Shape:
        # [B,S,hidden]
        #
        # ->
        #
        # [B,S,d_model]
        self.fc2 = nn.Linear(
            hidden,
            d_model,
            bias=False
        )

        # Dropout layer
        # Helps reduce overfitting
        self.dropout = nn.Dropout(dropout)

    # Forward pass through one expert
    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:

        # Apply first linear layer
        # Apply GELU activation
        # Apply dropout
        # Apply second linear layer
        return self.fc2(
            self.dropout(
                F.gelu(
                    self.fc1(x)
                )
            )
        )


# ==================================================
# DEEPSEEK MIXTURE OF EXPERTS (MoE)
# ==================================================
# Instead of sending every token through one large FFN,
# MoE routes tokens to a small number of experts.
#
# Benefits:
# 1. More model capacity
# 2. Lower computation cost
# 3. Expert specialization
#
# DeepSeek uses:
# - Shared Experts
# - Routed Experts
#
# Shared Experts:
# Process all tokens
#
# Routed Experts:
# Process only selected tokens

class DeepSeekMoE(nn.Module):

    # Constructor
    def __init__(self, args: ModelArgs):

        # Initialize parent module
        super().__init__()

        # Number of routed experts
        #
        # Example:
        # 8 experts
        self.n_routed = args.moe_n_routed_experts

        # Number of experts selected per token
        #
        # Example:
        # top_k = 2
        #
        # Each token uses only 2 experts
        self.top_k = args.moe_top_k

        # Create routed experts
        #
        # These experts receive selected tokens only
        self.routed_experts = nn.ModuleList(

            [
                ExpertFFN(
                    args.d_model,
                    args.moe_routed_hidden
                )

                for _ in range(self.n_routed)
            ]
        )

        # Create shared experts
        #
        # Shared experts process every token
        self.shared_experts = nn.ModuleList(

            [
                ExpertFFN(
                    args.d_model,
                    args.moe_routed_hidden
                )

                for _ in range(
                    args.moe_n_shared_experts
                )
            ]
        )

        # Router network
        #
        # Produces expert scores
        #
        # Input:
        # [token_embedding]
        #
        # Output:
        # score for each expert
        self.gate = nn.Linear(
            args.d_model,
            self.n_routed,
            bias=False
        )

        # Expert balancing bias
        #
        # Prevents expert collapse
        self.register_buffer(
            "bias",
            torch.zeros(self.n_routed)
        )

        # Learning rate for balancing bias
        self.bias_lr = 0.01


    # ==================================================
    # FORWARD PASS
    # ==================================================
    def forward(
        self,
        x: torch.Tensor
    ):

        # Input shape
        #
        # B = Batch Size
        # S = Sequence Length
        # D = Hidden Dimension
        B, S, D = x.shape

        # Flatten all tokens
        #
        # [B,S,D]
        #
        # ->
        #
        # [B*S,D]
        x_flat = x.reshape(-1, D)

        # Initialize shared output
        shared_out = torch.zeros_like(x)

        # ==================================================
        # SHARED EXPERTS
        # ==================================================
        # Every token passes through every shared expert
        for exp in self.shared_experts:

            shared_out += exp(x)

        # ==================================================
        # ROUTER
        # ==================================================

        # Compute routing scores
        #
        # Shape:
        # [B*S,n_routed]
        router_logits = self.gate(x_flat)

        # Add balancing bias
        router_logits_with_bias = (
            router_logits +
            self.bias.to(router_logits.dtype)
        )

        # Select Top-K experts
        #
        # top_k_logits:
        # Expert scores
        #
        # top_k_indices:
        # Expert IDs
        top_k_logits, top_k_indices = torch.topk(
            router_logits_with_bias,
            self.top_k,
            dim=-1
        )

        # Convert scores into probabilities
        #
        # Example:
        # [3.1,2.7]
        #
        # ->
        #
        # [0.60,0.40]
        gates = F.softmax(
            top_k_logits,
            dim=-1,
            dtype=torch.float
        ).type_as(x)

        # Storage for routed outputs
        routed_out_flat = torch.zeros_like(
            x_flat
        )

        # ==================================================
        # PROCESS EACH EXPERT
        # ==================================================
        for i in range(self.n_routed):

            # Find tokens assigned to expert i
            mask = (top_k_indices == i)

            # Get token indices
            row_idx, which_k = torch.where(mask)

            # Skip if no tokens assigned
            if row_idx.numel() == 0:
                continue

            # Routing weights
            w = gates[
                row_idx,
                which_k
            ].unsqueeze(-1)

            # Extract tokens
            exp_in = x_flat[row_idx]

            # Process tokens through expert
            exp_out = self.routed_experts[i](
                exp_in
            )

            # Add weighted outputs
            routed_out_flat.index_add_(
                0,
                row_idx,
                exp_out * w
            )

        # ==================================================
        # LOAD BALANCING
        # ==================================================
        # Prevent one expert from receiving
        # all tokens while others remain unused.
        if self.training:

            with torch.no_grad():

                # Ideal number of tokens per expert
                avg_load = (
                    x_flat.size(0)
                    * self.top_k
                    / self.n_routed
                )

                # Count assigned tokens
                counts = torch.bincount(
                    top_k_indices.flatten(),
                    minlength=self.n_routed
                ).float()

                # Load imbalance
                violation = (
                    avg_load - counts
                ) / (
                    avg_load + 1e-6
                )

                # Update balancing bias
                self.bias.add_(
                    self.bias_lr *
                    torch.tanh(violation)
                )

        # ==================================================
        # RESTORE ORIGINAL SHAPE
        # ==================================================

        # [B*S,D]
        #
        # ->
        #
        # [B,S,D]
        routed_out = routed_out_flat.view(
            B,
            S,
            D
        )

        # Final output
        #
        # Shared Expert Output
        # +
        # Routed Expert Output
        return (
            shared_out +
            routed_out
        )

# Multi-Token Prediction (MTP) Module

In [7]:
# ==================================================
# MULTI-TOKEN PREDICTION (MTP) MODULE
# ==================================================
# MTP allows the model to predict multiple future tokens
# during training instead of predicting only the next token.
#
# Standard GPT:
# Predicts Token t+1
#
# MTP:
# Predicts Token t+1
# Predicts Token t+2
# Predicts Token t+3
#
# This improves training efficiency and helps the model
# learn longer-range dependencies.
#
# DeepSeek-V3 uses MTP as an auxiliary training objective.

class MTPModule(nn.Module):

    # Constructor
    def __init__(self, args: ModelArgs):

        # Initialize parent PyTorch module
        super().__init__()

        # Projection layer
        #
        # Concatenates:
        # Previous hidden state
        # +
        # Next token embedding
        #
        # Input:
        # [B,S,2*d_model]
        #
        # Output:
        # [B,S,d_model]
        self.projection = nn.Linear(
            args.d_model * 2,
            args.d_model,
            bias=False
        )

        # Internal Transformer block
        # Used to predict future tokens
        self.block = TransformerBlock(args)

    # Forward Pass
    def forward(
        self,
        h_prev,
        next_token_embeds
    ):

        # Concatenate hidden state and future token embedding
        #
        # [B,S,D]
        # +
        # [B,S,D]
        #
        # ->
        #
        # [B,S,2D]
        x = torch.cat(
            [h_prev, next_token_embeds],
            dim=-1
        )

        # Project back to model dimension
        #
        # [B,S,2D]
        #
        # ->
        #
        # [B,S,D]
        x = self.projection(x)

        # Extract dimensions
        B, S, D = x.shape

        # ==================================================
        # CREATE CAUSAL MASK
        # ==================================================
        # Prevent future token leakage
        #
        # Example:
        #
        # 0  -∞ -∞
        # 0   0 -∞
        # 0   0  0
        mask = torch.triu(
            torch.ones(
                S,
                S,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # Convert boolean mask to attention mask
        attn_mask = torch.zeros(
            S,
            S,
            device=x.device
        ).masked_fill(
            mask,
            float('-inf')
        )

        # Run through Transformer block
        #
        # No KV Cache used during MTP training
        h_k, _ = self.block(
            x,
            attn_mask=attn_mask
                .unsqueeze(0)
                .unsqueeze(0)
        )

        # Return future prediction hidden states
        return h_k


# ==================================================
# TRANSFORMER BLOCK
# ==================================================
# One complete DeepSeek Transformer layer.
#
# Structure:
#
# Input
#   ↓
# LayerNorm
#   ↓
# MLA Attention
#   ↓
# Residual Connection
#   ↓
# LayerNorm
#   ↓
# MoE Feed Forward
#   ↓
# Residual Connection
#   ↓
# Output

class TransformerBlock(nn.Module):

    # Constructor
    def __init__(self, args: ModelArgs):

        # Initialize parent module
        super().__init__()

        # First LayerNorm
        self.norm1 = nn.LayerNorm(
            args.d_model
        )

        # Multi-Head Latent Attention
        self.attention = DeepSeekAttention(
            args
        )

        # Second LayerNorm
        self.norm2 = nn.LayerNorm(
            args.d_model
        )

        # Mixture of Experts
        self.feed_forward = DeepSeekMoE(
            args
        )

    # Forward Pass
    def forward(
        self,
        x: torch.Tensor,
        attn_mask: torch.Tensor,
        past_kv=None,
        position_offset: int = 0
    ):

        # ==================================================
        # MLA ATTENTION
        # ==================================================

        # Normalize input
        normalized_x = self.norm1(x)

        # Run attention
        h, new_cache = self.attention(
            normalized_x,
            attn_mask,
            past_kv,
            position_offset
        )

        # Residual connection
        x = x + h

        # ==================================================
        # MOE FEED FORWARD
        # ==================================================

        # Normalize hidden states
        normalized_x = self.norm2(x)

        # Apply MoE
        moe_output = self.feed_forward(
            normalized_x
        )

        # Residual connection
        x = x + moe_output

        # Return hidden state and cache
        return x, new_cache


# ==================================================
# MAIN MODEL : MINI DEEPSEEK
# ==================================================
# Full architecture:
#
# Input IDs
#     ↓
# Token Embedding
#     ↓
# N Transformer Blocks
#     ↓
# Final LayerNorm
#     ↓
# LM Head
#     ↓
# Vocabulary Logits

class MiniDeepSeek(nn.Module):

    # Constructor
    def __init__(self, args: ModelArgs):

        # Initialize parent module
        super().__init__()

        # Save configuration
        self.args = args

        # ==================================================
        # TOKEN EMBEDDING
        # ==================================================
        #
        # Converts token IDs into vectors
        #
        # Example:
        #
        # Token 245
        #
        # ->
        #
        # [0.12, -0.45, ...]
        self.embed = nn.Embedding(
            args.vocab_size,
            args.d_model
        )

        # ==================================================
        # TRANSFORMER STACK
        # ==================================================
        #
        # Create multiple Transformer blocks
        self.blocks = nn.ModuleList(

            [
                TransformerBlock(args)

                for _ in range(
                    args.n_layers
                )
            ]
        )

        # Final normalization layer
        self.norm_f = nn.LayerNorm(
            args.d_model
        )

        # Language modeling head
        #
        # Converts hidden states into
        # vocabulary prediction scores
        self.lm_head = nn.Linear(
            args.d_model,
            args.vocab_size,
            bias=False
        )

        # MTP Modules
        #
        # Used only during training
        self.mtp_modules = nn.ModuleList(

            [
                MTPModule(args)

                for _ in range(
                    args.n_mtp_modules
                )
            ]
        )

    # ==================================================
    # CAUSAL MASK
    # ==================================================
    # Prevents future token access
    def causal_mask(
        self,
        S: int,
        device
    ) -> torch.Tensor:

        # Create upper triangular mask
        mask = torch.triu(
            torch.ones(
                S,
                S,
                device=device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # Convert mask into -inf values
        return torch.zeros(
            S,
            S,
            device=device
        ).masked_fill(
            mask,
            float('-inf')
        )
        
    # ==================================================
    # FORWARD PASS
    # ==================================================
    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None
    ):
        

        # Batch size and sequence length
        B, S = idx.shape

        # Convert token IDs into embeddings
        x = self.embed(idx)

        # Create causal attention mask
        attn_mask = self.causal_mask(
            S,
            idx.device
        ).unsqueeze(0).unsqueeze(0)

        # Pass through all Transformer blocks
        for block in self.blocks:
            x, _ = block(
                x,
                attn_mask,
            )

        # Apply final LayerNorm
        x = self.norm_f(x)

        # Compute vocabulary logits
        logits = self.lm_head(x)

        # Initialize loss
        loss = None

        # Compute cross-entropy loss during training
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )

        # Return outputs
        return {
            "logits": logits,
            "loss": loss
        }

In [8]:
import gc
import torch

gc.collect()

torch.cuda.empty_cache()

if torch.cuda.is_available():
    torch.cuda.ipc_collect()

print("GPU cache cleared.")

GPU cache cleared.


In [9]:
import gc
gc.collect()

torch.cuda.empty_cache()

if torch.cuda.is_available():
    torch.cuda.ipc_collect()

# Train

In [10]:
# ==================================================
# TRAINING SCRIPT (train.ipynb)
# ==================================================
# This script trains the MiniDeepSeek model on the
# tokenized TinyStories dataset.
#
# Training Pipeline:
#
# Load Tokenized Data
#          ↓
# Create MiniDeepSeek Model
#          ↓
# Create Optimizer
#          ↓
# Forward Pass
#          ↓
# Compute Loss
#          ↓
# Backpropagation
#          ↓
# Update Parameters
#          ↓
# Validation
#          ↓
# Save Checkpoints

! pip install torch
# ==================================================
# IMPORT LIBRARIES
# ==================================================

# File and directory operations
import os

# Read metadata JSON file
import json

# Timing utilities
import time

# Provides null context manager
# Used when mixed precision is disabled
from contextlib import nullcontext

# Mathematical functions
# Used by cosine learning rate scheduler
import math

# Numerical array library
import numpy as np

# PyTorch framework
import torch

# Neural network functions
from torch.nn import functional as F

# ==================================================
# SYSTEM CONFIGURATION
# ==================================================

# Use GPU if available
device = (
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)

# Choose numerical precision
dtype = (

    'bfloat16'

    if torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()

    else

    'float16'
)

# Enable TensorFloat-32 matrix multiplication
# Improves speed on NVIDIA GPUs
torch.backends.cuda.matmul.allow_tf32 = True

# Enable TF32 inside cuDNN
torch.backends.cudnn.allow_tf32 = True

# Convert datatype string to PyTorch datatype
pt_dtype = {

    'float32': torch.float32,

    'bfloat16': torch.bfloat16,

    'float16': torch.float16

}[dtype]

# Automatic Mixed Precision Context
ctx = (

    torch.amp.autocast(
        device_type=device.split(':')[0],
        dtype=pt_dtype
    )

    if 'cuda' in device

    else

    nullcontext()
)


# ==================================================
# TRAINING CONFIGURATION
# ==================================================

# Output directory
out_dir = 'out'

# Location of tokenized dataset
data_dir = 'data/tinystories_tokenized'

# Total training iterations
max_iters = 5000

# Validation frequency
eval_interval = 250

# Logging frequency
log_interval = 20

# Number of validation batches
eval_iters = 100

# Batch size
#
# Number of sequences processed together
batch_size = 4

# Context window length
#
# Number of tokens per sequence
block_size = 128


# ==================================================
# ADAMW OPTIMIZER SETTINGS
# ==================================================

# Initial learning rate
learning_rate = 4e-4

# Weight decay regularization
weight_decay = 0.1

# Adam momentum coefficient
beta1 = 0.9

# Adam variance coefficient
beta2 = 0.95


# ==================================================
# DATA LOADER
# ==================================================
# Loads batches directly from binary files.

def get_batch(split: str):

    """
    Loads a batch of tokenized sequences.
    """

    # Open binary token file
    #
    # train.bin
    # or
    # val.bin
    data = np.memmap(

        os.path.join(
            data_dir,
            f'{split}.bin'
        ),

        dtype=np.uint16,

        mode='r'
    )

    # Generate random starting positions
    ix = torch.randint(

        len(data) - block_size,

        (batch_size,)
    )

    # ==================================================
    # CREATE INPUT TOKENS
    # ==================================================
    #
    # Example:
    #
    # Input:
    # [10,20,30,40]
    #
    # Target:
    # [20,30,40,50]

    x = torch.stack(

        [

            torch.from_numpy(

                (
                    data[i:i+block_size]
                ).astype(np.int64)

            )

            for i in ix
        ]
    )

    # Create target tokens
    y = torch.stack(

        [

            torch.from_numpy(

                (
                    data[
                        i+1:
                        i+1+block_size
                    ]
                ).astype(np.int64)

            )

            for i in ix
        ]
    )

    # ==================================================
    # MOVE DATA TO GPU
    # ==================================================

    if 'cuda' in device:

        return (

            x.pin_memory().to(
                device,
                non_blocking=True
            ),

            y.pin_memory().to(
                device,
                non_blocking=True
            )
        )

    else:

        return x, y


# ==================================================
# COSINE LEARNING RATE SCHEDULER
# ==================================================
#
# Phase 1:
# Warmup
#
# Phase 2:
# Cosine Decay

def get_lr(it):

    # Warmup length
    warmup_iters = 200

    # ==================================================
    # LINEAR WARMUP
    # ==================================================

    if it < warmup_iters:

        return (
            learning_rate *
            it /
            warmup_iters
        )

    # ==================================================
    # MINIMUM LEARNING RATE
    # ==================================================

    lr_decay_iters = max_iters

    min_lr = (
        learning_rate / 10
    )

    if it > lr_decay_iters:

        return min_lr

    # ==================================================
    # COSINE DECAY
    # ==================================================

    decay_ratio = (

        (it - warmup_iters)

        /

        (lr_decay_iters - warmup_iters)
    )

    assert 0 <= decay_ratio <= 1

    coeff = 0.5 * (

        1.0 +

        math.cos(
            math.pi *
            decay_ratio
        )
    )

    return (

        min_lr +

        coeff *

        (
            learning_rate -
            min_lr
        )
    )


# ==================================================
# MAIN TRAINING SCRIPT
# ==================================================

if __name__ == '__main__':

    # Create output folder
    os.makedirs(
        out_dir,
        exist_ok=True
    )

    # ==================================================
    # LOAD METADATA
    # ==================================================

    meta_path = os.path.join(
        data_dir,
        'meta.json'
    )

    try:

        with open(
            meta_path,
            'r'
        ) as f:

            meta = json.load(f)

        vocab_size = meta[
            'vocab_size'
        ]

    except FileNotFoundError:

        print(
            f"Error: meta.json not found "
            f"in {data_dir}."
        )

        print(
            "Please run prepare.py first."
        )

        exit(1)


    # ==================================================
    # MODEL CONFIGURATION
    # ==================================================
    #
    # Approximately 130 Million Parameters

    model_args = ModelArgs(

        # Hidden dimension
        d_model=768,

        # Number of Transformer blocks
        n_layers=12,

        # Number of attention heads
        num_heads=12,

        # MLA latent dimension
        d_latent=192,

        # RoPE dimension
        d_rope=64,

        vocab_size=vocab_size,
        
        max_seq_len=block_size

    )

    # Create MiniDeepSeek model
    model = MiniDeepSeek(
        model_args
    )

    # Move model to GPU
    model.to(device)


    # ==================================================
    # PARAMETER STATISTICS
    # ==================================================

    total_params = sum(

        p.numel()

        for p in model.parameters()
    )

    trainable_params = sum(

        p.numel()

        for p in model.parameters()

        if p.requires_grad
    )

    print(
        f"\nModel initialized on {device}"
    )

    print(
        f"  -> Total parameters: "
        f"{total_params/1e6:.2f}M"
    )

    print(
        f"  -> Trainable parameters: "
        f"{trainable_params/1e6:.2f}M\n"
    )


    # ==================================================
    # CREATE OPTIMIZER
    # ==================================================

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=learning_rate,

        weight_decay=weight_decay,

        betas=(beta1, beta2)
    )


    # ==================================================
    # TRAINING LOOP
    # ==================================================

    t0 = time.time()

    best_val_loss = float('inf')

    for iter_num in range(max_iters):

        # ------------------------------------
        # UPDATE LEARNING RATE
        # ------------------------------------

        lr = get_lr(iter_num)

        for param_group in optimizer.param_groups:

            param_group['lr'] = lr


        # ------------------------------------
        # TRAINING MODE
        # ------------------------------------

        model.train()

        # Load training batch
        X, Y = get_batch('train')

        # Forward pass
        with ctx:

            outputs = model(
                X,
                targets=Y
            )

            loss = outputs['loss']

        # Backpropagation
        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()


        # ------------------------------------
        # LOGGING
        # ------------------------------------

        if iter_num % log_interval == 0:

            t1 = time.time()

            dt = t1 - t0

            t0 = t1

            print(
                f"iter {iter_num}: "
                f"loss {loss.item():.4f}, "
                f"time {dt*1000:.2f}ms, "
                f"lr {lr:.6f}"
            )


        # ------------------------------------
        # VALIDATION
        # ------------------------------------

        if (
            iter_num % eval_interval == 0
            and
            iter_num > 0
        ):

            model.eval()

            losses = torch.zeros(
                eval_iters
            )

            print(
                "Running evaluation..."
            )

            with torch.no_grad():

                for k in range(eval_iters):

                    X, Y = get_batch('val')

                    with ctx:

                        outputs = model(
                            X,
                            targets=Y
                        )

                        losses[k] = (
                            outputs['loss']
                            .item()
                        )

            val_loss = losses.mean()

            print(
                f"iter {iter_num}: "
                f"validation loss "
                f"{val_loss:.4f}"
            )

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.26a0+c5e1555-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is depreca

# Sample

In [11]:
# ==================================================
# TEXT GENERATION SCRIPT (sample.py)
# ==================================================
# This script loads a trained MiniDeepSeek model
# and generates text from a user-provided prompt.
#
# Workflow:
#
# Load Checkpoint
#         ↓
# Load Model
#         ↓
# Load Tokenizer
#         ↓
# User Prompt
#         ↓
# Convert Text → Tokens
#         ↓
# Generate Next Token
#         ↓
# Update KV Cache
#         ↓
# Repeat Until Max Tokens
#         ↓
# Display Generated Text


# ==================================================
# IMPORT LIBRARIES
# ==================================================

# Provides file and directory operations
import os

# Used for reading JSON data
import json

# PyTorch deep learning framework
import torch

# GPT tokenizer library
import tiktoken

# Import PyTorch serialization module
# Required for secure checkpoint loading
from torch import serialization


# ==================================================
# CONFIGURATION
# ==================================================

# Directory containing trained checkpoints
out_dir = 'out'

# Use GPU if available
# Otherwise use CPU
device = (
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)

# Choose best floating-point precision
#
# BF16:
# Faster and memory efficient
#
# FP16:
# Fallback if BF16 unavailable
dtype = (
    'bfloat16'
    if torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
    else 'float16'
)

# Convert string datatype into PyTorch datatype
pt_dtype = {

    'float32': torch.float32,

    'bfloat16': torch.bfloat16,

    'float16': torch.float16

}[dtype]

# Automatic Mixed Precision Context
#
# GPU:
# Uses autocast
#
# CPU:
# Uses no_grad
ctx = (

    torch.amp.autocast(
        device_type=device.split(':')[0],
        dtype=pt_dtype
    )

    if 'cuda' in device

    else

    torch.no_grad()
)


# ==================================================
# CHECKPOINT LOADER
# ==================================================
# Finds the newest checkpoint and loads it.

def load_latest_checkpoint(
    directory: str
):

    # Documentation string
    """Loads the latest checkpoint from the specified directory."""

    # Find all checkpoint files
    files = [

        f

        for f in os.listdir(directory)

        if f.startswith('ckpt_iter_')
        and f.endswith('.pt')
    ]

    # Return None if no checkpoints found
    if not files:

        return None

    # Select checkpoint with largest iteration number
    latest_file = max(

        files,

        key=lambda f: int(
            f.split('_')[-1]
             .split('.')[0]
        )
    )

    # Build full path
    ckpt_path = os.path.join(
        directory,
        latest_file
    )

    # Show selected checkpoint
    print(
        f"Loading checkpoint: {ckpt_path}"
    )

    # ==================================================
    # PYTORCH 2.6 SECURITY FEATURE
    # ==================================================
    #
    # PyTorch 2.6 requires explicit declaration
    # of safe classes during unpickling.
    #
    # Our checkpoint stores:
    #
    # ModelArgs
    #
    # so we whitelist it here.
    with serialization.safe_globals(
        [ModelArgs]
    ):

        checkpoint = torch.load(
            ckpt_path,
            map_location=device
        )

    # Return checkpoint dictionary
    return checkpoint

# Main Sample 

In [ ]:
def main():

    # Switch to inference mode
    model.eval()

    # Display status
    print(
        "Model loaded successfully."
    )

    print("-" * 50)


    # ==================================================
    # LOAD TOKENIZER
    # ==================================================

    # GPT-2 tokenizer
    enc = tiktoken.get_encoding(
        "gpt2"
    )


    # ==================================================
    # INTERACTIVE GENERATION LOOP
    # ==================================================
    #
    # Runs forever until Ctrl+C

    while True:

        try:

            # Ask user for prompt
            start_text = input(
                "Enter a prompt "
                "(or press Ctrl+C to exit): "
            )

            # Skip empty prompts
            if not start_text:

                continue

            # ==================================================
            # TOKENIZE PROMPT
            # ==================================================

            # Convert text to token IDs
            start_ids = enc.encode(
                start_text
            )

            # Convert tokens to tensor
            x = torch.tensor(

                start_ids,

                dtype=torch.long,

                device=device

            ).unsqueeze(0)

            # Display prompt
            print(
                f"Prompt: '{start_text}'"
            )

            print(
                "Generating: ",
                end='',
                flush=True
            )

            # ==================================================
            # INITIALIZE KV CACHE
            # ==================================================

            kv_cache = None

            # Maximum generated tokens
            max_new_tokens = 100


            # ==================================================
            # AUTOREGRESSIVE GENERATION
            # ==================================================

            with torch.no_grad():

                with ctx:

                    # Generate tokens one-by-one
                    for _ in range(
                        max_new_tokens
                    ):

                        # Forward pass
                        outputs = model(x)

                        logits = outputs["logits"]

                        # Get last token logits
                        logits = logits[:, -1, :]

                        # Greedy decoding
                        #
                        # Select highest probability token
                        next_token = torch.argmax(

                            logits,

                            dim=-1
                        )

                        # Prepare next input
                        #
                        # Shape:
                        # [1]
                        #
                        # ->
                        #
                        # [1,1]
                        x = torch.cat(
                        [x, next_token.unsqueeze(1)],
                        dim=1
                       )

                        # Convert token to text
                        token_str = enc.decode(

                            [next_token.item()]
                        )

                        # Print generated token
                        print(

                            token_str,

                            end='',

                            flush=True
                        )

            # Finished generation
            print(
                "\n" + "-" * 50
            )

        # ==================================================
        # EXIT HANDLER
        # ==================================================
        except KeyboardInterrupt:

            print(
                "\nExiting."
            )

            break


# ==================================================
# SCRIPT ENTRY POINT
# ==================================================

if __name__ == '__main__':

    # Start generation script
    main()

Model loaded successfully.
--------------------------------------------------


Enter a prompt (or press Ctrl+C to exit):  Once upon a time


Prompt: 'Once upon a time'
, there was a little girl named Lily. She loved to play outside in the sun. One day, she saw a big, red ball. It was a big, red ball. Lily wanted to play with it, but her mom said it was too late.

Lily was sad and didn't know what to do. She asked her mom if she could play with it. Her mom said yes and they went to the park. Lily was happy and thanked her mom for helping her.
--------------------------------------------------


Enter a prompt (or press Ctrl+C to exit):  
